# Legacy vs. shared-engine benchmark
This notebook keeps the comparison auditable: each table is the input to the next calculation. It compares active query latency, whole-workload performance, and observed peak memory at matched concurrency and aggregate buffer capacity.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

# Change only these paths when analyzing a new pair of runs.
LEGACY_SUMMARY = Path("results/legacy-multi-jvm/all_summary.csv")
LEGACY_QUERIES = Path("results/legacy-multi-jvm/all_raw.csv")
SHARED_RUN = Path("results/shared-engine/REPLACE_WITH_RUN_DIRECTORY")
LEGACY_RUN_ID = None  # None selects the newest run in the append-only CSVs.
LEGACY_SUMMARY, SHARED_RUN

## 1. Load and inspect the runner outputs
The legacy aggregate files contain several runs, so the cells select one run ID. The shared runner writes one set of CSVs per run directory. Failed runner invocations are not valid notebook inputs.

In [ ]:
legacy_summary_all = pd.read_csv(LEGACY_SUMMARY)
legacy_queries_all = pd.read_csv(LEGACY_QUERIES)
shared_summary = pd.read_csv(SHARED_RUN / "summary.csv")
shared_queries = pd.read_csv(SHARED_RUN / "queries.csv")
shared_repetitions = pd.read_csv(SHARED_RUN / "repetitions.csv")

run_id = LEGACY_RUN_ID or legacy_summary_all.sort_values("run_created_at_utc").iloc[-1]["run_id"]
legacy_summary = legacy_summary_all[legacy_summary_all.run_id == run_id].copy()
legacy_queries = legacy_queries_all[legacy_queries_all.run_id == run_id].copy()
print(f"Legacy run: {run_id}; shared run: {SHARED_RUN.name}")
display(legacy_queries.head(3), shared_queries.head(3))

## 2. Check that the runs answer the same question
Only the load-bearing conditions are checked here: workload definitions, concurrency levels, and total buffer frames. Environment equivalence is controlled manually when the benchmarks run.

In [ ]:
assert (legacy_summary.failed == 0).all(), "Legacy run contains failed queries"
assert (shared_summary.failed == 0).all(), "Shared run contains failed queries"
# Inspect one repetition: repeated workload rows are meaningful and remain visible.
legacy_first = legacy_queries[(legacy_queries.concurrency == legacy_queries.concurrency.min()) & (legacy_queries.repetition == legacy_queries.repetition.min())]
shared_first = shared_queries[(shared_queries.max_concurrent == shared_queries.max_concurrent.min()) & (shared_queries.repetition == shared_queries.repetition.min())]
workload_columns = ["workload", "start_range", "end_range"]
legacy_workload = legacy_first[workload_columns].sort_values(workload_columns).reset_index(drop=True)
shared_workload = shared_first[workload_columns].sort_values(workload_columns).reset_index(drop=True)
assert legacy_workload.equals(shared_workload), "Workload rows or their multiplicities differ"

legacy_config = (legacy_summary.groupby("concurrency", as_index=False).agg(legacy_buffer_per_jvm=("buffer_size", "first")))
legacy_config["legacy_total_buffer"] = legacy_config.concurrency * legacy_config.legacy_buffer_per_jvm
shared_config = shared_summary[["max_concurrent", "buffer_size"]].rename(columns={"max_concurrent": "concurrency", "buffer_size": "shared_total_buffer"})
comparability = legacy_config.merge(shared_config, on="concurrency", how="outer", indicator=True)
comparability["buffers_match"] = comparability.legacy_total_buffer == comparability.shared_total_buffer
display(legacy_workload, comparability)
assert (comparability._merge == "both").all(), "Concurrency levels differ"
assert comparability.buffers_match.all(), "Aggregate buffer allocations differ"

## 3. Align active query latency
Legacy `query_elapsed_ms` and shared `execution_ms` both exclude admission waiting. The table first shows the matched observations, then averages repetitions for each workload and concurrency.

In [ ]:
legacy_latency = legacy_queries[["concurrency", "repetition", "workload", "query_elapsed_ms"]].rename(columns={"query_elapsed_ms": "legacy_ms"})
shared_latency = shared_queries[["max_concurrent", "repetition", "workload", "execution_ms"]].rename(columns={"max_concurrent": "concurrency", "execution_ms": "shared_ms"})
# Normalize repetition numbering, then distinguish repeated copies of a workload.
legacy_latency["repetition"] -= legacy_latency.repetition.min()
shared_latency["repetition"] -= shared_latency.repetition.min()
keys = ["concurrency", "repetition", "workload"]
legacy_latency["occurrence"] = legacy_latency.groupby(keys).cumcount()
shared_latency["occurrence"] = shared_latency.groupby(keys).cumcount()
aligned_queries = legacy_latency.merge(shared_latency, on=keys + ["occurrence"], validate="one_to_one")
assert len(aligned_queries) == len(legacy_latency) == len(shared_latency), "Query observations are incomplete"
display(aligned_queries.head(10))
query_comparison = aligned_queries.groupby(["concurrency", "workload"], as_index=False)[["legacy_ms", "shared_ms"]].mean()
query_comparison["shared_speedup"] = query_comparison.legacy_ms / query_comparison.shared_ms
display(query_comparison.round(2))

In [ ]:
for concurrency, rows in query_comparison.groupby("concurrency"):
    rows.plot(x="workload", y=["legacy_ms", "shared_ms"], kind="bar", title=f"Active query latency - concurrency {concurrency}", ylabel="milliseconds")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 4. Compare the whole workload
Makespan includes the effect of bounded concurrency on the batch. Throughput is completed queries divided by makespan.

In [ ]:
legacy_workload_results = legacy_summary.groupby("concurrency", as_index=False).agg(legacy_makespan_ms=("makespan_seconds", lambda x: x.mean() * 1000), legacy_qps=("throughput_qps", "mean"), legacy_peak_mb=("aggregate_peak_rss_mb", "max"))
shared_workload_results = shared_repetitions.groupby("max_concurrent", as_index=False).agg(shared_makespan_ms=("makespan_ms", "mean"), shared_qps=("throughput_qps", "mean")).rename(columns={"max_concurrent": "concurrency"})
workload_comparison = legacy_workload_results.merge(shared_workload_results, on="concurrency", validate="one_to_one")
workload_comparison["throughput_gain"] = workload_comparison.shared_qps / workload_comparison.legacy_qps
workload_comparison["makespan_speedup"] = workload_comparison.legacy_makespan_ms / workload_comparison.shared_makespan_ms
display(workload_comparison.drop(columns="legacy_peak_mb").round(2))
workload_comparison.plot(x="concurrency", y=["legacy_makespan_ms", "shared_makespan_ms"], marker="o", ylabel="milliseconds", title="Mean workload makespan")
plt.show()
workload_comparison.plot(x="concurrency", y=["legacy_qps", "shared_qps"], marker="o", ylabel="queries / second", title="Mean workload throughput")
plt.show()

## 5. Compare observed peak memory
Legacy memory is the maximum of the per-repetition aggregate process-group peaks. Shared memory is the process-group maximum recorded for the configuration JVM across warmup and measured repetitions. Both values are maxima, but their sampling windows still differ, so this is an observed-envelope comparison rather than identical measurement periods.

In [ ]:
shared_peaks = []
for config in SHARED_RUN.glob("concurrency-*-buffer-*"):
    resource = json.loads((config / "resources.json").read_text())
    shared_peaks.append({"concurrency": int(config.name.split("-")[1]), "shared_peak_mb": resource["monitor"]["aggregate_peak_rss_bytes"] / 1024**2})
memory_comparison = workload_comparison[["concurrency", "legacy_peak_mb"]].merge(pd.DataFrame(shared_peaks), on="concurrency", validate="one_to_one")
memory_comparison["memory_reduction_pct"] = (1 - memory_comparison.shared_peak_mb / memory_comparison.legacy_peak_mb) * 100
display(memory_comparison.round(2))
memory_comparison.plot(x="concurrency", y=["legacy_peak_mb", "shared_peak_mb"], kind="bar", ylabel="MiB", title="Observed aggregate peak RSS")
plt.show()

## Findings and limitations
Use the displayed speedup and memory-reduction columns to write the result after running the notebook. Claims apply only to the selected workload, Amazon Linux host, and concurrency levels. The experiment does not establish linear scalability or isolate JVM startup, cache sharing, and buffer-manager contention as separate causes. Shared admission wait remains diagnostic data and is intentionally excluded from active query latency.